#01 - Gold Layer - Sales Fact Table

Create the central business-ready Sales Fact table from validated Silver transactions.

**Source:** end-to-end_pipeline.silver.sales_transactions

**Target:** end-to-end_pipeline.gold.fact_sales

**Model Role:** Fact Table

**Grain:** One row per sales transaction line

**Business Key:** transaction_id

**Approach:** Profile → Inspect → Transform → Validate

**Purpose:**
Provide the central transactional table for sales analytics. The table connects sales transactions to Customer, Product, Store, and Date dimensions and provides reusable row-level business measures for revenue, discount, cost, cancellation, and profit analysis.

#Cell 1 - Profile Silver Sales Data

**Description:**
Confirm that the Silver transaction table is ready for Gold modeling. This checks transaction grain, key uniqueness, status distribution, date coverage, and the numeric fields required to calculate business measures.

In [0]:
%sql

-- ============================================================
-- CELL 1: PROFILE SILVER SALES FOR GOLD MODELING
-- Purpose: Confirm transaction grain, key uniqueness,
--          date coverage, and measure availability
-- ============================================================

SELECT
    COUNT(*) AS total_rows,

    COUNT(DISTINCT transaction_id)
        AS distinct_transaction_ids,

    COUNT(*) - COUNT(DISTINCT transaction_id)
        AS duplicate_transaction_ids,

    SUM(CASE WHEN transaction_id IS NULL THEN 1 ELSE 0 END)
        AS null_transaction_ids,

    SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END)
        AS null_order_ids,

    SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END)
        AS null_customer_ids,

    SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END)
        AS null_product_ids,

    SUM(CASE WHEN store_id IS NULL THEN 1 ELSE 0 END)
        AS null_store_ids,

    COUNT(DISTINCT order_status)
        AS order_statuses,

    MIN(order_date)
        AS earliest_order_date,

    MAX(order_date)
        AS latest_order_date,

    MIN(quantity)
        AS minimum_quantity,

    MAX(quantity)
        AS maximum_quantity

FROM `end-to-end_pipeline`.silver.sales_transactions;

total_rows,distinct_transaction_ids,duplicate_transaction_ids,null_transaction_ids,null_order_ids,null_customer_ids,null_product_ids,null_store_ids,order_statuses,earliest_order_date,latest_order_date,minimum_quantity,maximum_quantity
24905,24905,0,0,0,0,0,0,3,2021-01-01,2025-12-31,1,5


#Cell 2 - Inspect Business and Dimension Relationships

**Description:**
Review transaction status distribution and check whether the Sales Fact keys can connect correctly to the Gold dimensions.

This does not modify the data. It helps identify possible unmatched Customer, Product, Store, or Date keys before the fact table is used by dashboards and Genie.

In [0]:
%sql

-- ============================================================
-- CELL 2: INSPECT BUSINESS STATUS AND DIMENSION RELATIONSHIPS
-- Purpose: Review transaction status and identify
--          unmatched dimension keys
-- ============================================================

SELECT
    s.order_status,
    COUNT(*) AS transaction_count,

    SUM(
        CASE
            WHEN c.customer_id IS NULL THEN 1
            ELSE 0
        END
    ) AS unmatched_customers,

    SUM(
        CASE
            WHEN p.product_id IS NULL THEN 1
            ELSE 0
        END
    ) AS unmatched_products,

    SUM(
        CASE
            WHEN st.store_id IS NULL THEN 1
            ELSE 0
        END
    ) AS unmatched_stores,

    SUM(
        CASE
            WHEN d.date IS NULL THEN 1
            ELSE 0
        END
    ) AS unmatched_dates

FROM `end-to-end_pipeline`.silver.sales_transactions s

LEFT JOIN `end-to-end_pipeline`.gold.dim_customer c
    ON s.customer_id = c.customer_id

LEFT JOIN `end-to-end_pipeline`.gold.dim_product p
    ON s.product_id = p.product_id

LEFT JOIN `end-to-end_pipeline`.gold.dim_store st
    ON s.store_id = st.store_id

LEFT JOIN `end-to-end_pipeline`.gold.dim_date d
    ON s.order_date = d.date

GROUP BY
    s.order_status

ORDER BY
    s.order_status;

order_status,transaction_count,unmatched_customers,unmatched_products,unmatched_stores,unmatched_dates
Cancelled,1470,1,5,0,0
Completed,22468,2,98,0,0
Returned,967,0,1,0,0


#Cell 3 - Transform Silver → Gold Sales Fact

**Description:**
Create the Gold Sales Fact table at one row per transaction.

This step preserves the transaction and dimension keys and adds reusable row-level business measures.

**Business measures**

* **gross_revenue** = Quantity × Unit Price
* **discount_amount** = Gross Revenue × Discount %
* **net_revenue** = Completed sales minus returned sales; cancelled transactions contribute zero revenue
* **total_cost** = Cost associated with completed/returned sales
* **profit** = Net Revenue − Total Cost
* **cancelled_value** = Value of cancelled transactions

In [0]:
%sql

-- ============================================================
-- CELL 3: CREATE GOLD SALES FACT TABLE
-- Grain: One row per sales transaction
-- Business Key: transaction_id
-- ============================================================

CREATE OR REPLACE TABLE `end-to-end_pipeline`.gold.fact_sales
COMMENT 'Business-ready sales fact table at transaction level. Provides sales, revenue, discount, cost, cancellation, and profit measures for Databricks dashboards and Genie.'
AS

WITH calculated AS (

    SELECT
        transaction_id,
        order_id,
        order_date,

        -- Dimension keys
        customer_id,
        product_id,
        store_id,

        -- Transaction attributes
        quantity,
        unit_price,
        unit_cost,
        discount_pct,
        order_status,
        payment_method,

        -- Gross value before discount
        CAST(
            quantity * unit_price
            AS DECIMAL(14,2)
        ) AS gross_revenue,

        -- Monetary value of discount
        CAST(
            quantity * unit_price * discount_pct
            AS DECIMAL(14,2)
        ) AS discount_amount

    FROM `end-to-end_pipeline`.silver.sales_transactions
)

SELECT
    transaction_id,
    order_id,
    order_date,

    customer_id,
    product_id,
    store_id,

    quantity,
    unit_price,
    unit_cost,
    discount_pct,

    order_status,
    payment_method,

    gross_revenue,
    discount_amount,

    -- Completed sales are positive revenue.
    -- Returns reverse previously recognized revenue.
    -- Cancelled orders generate no realized revenue.
    CAST(
        CASE
            WHEN order_status = 'Completed'
                THEN gross_revenue - discount_amount

            WHEN order_status = 'Returned'
                THEN -(gross_revenue - discount_amount)

            ELSE 0
        END
        AS DECIMAL(14,2)
    ) AS net_revenue,

    -- Cost follows the same realized-sale logic
    CAST(
        CASE
            WHEN order_status = 'Completed'
                THEN quantity * unit_cost

            WHEN order_status = 'Returned'
                THEN -(quantity * unit_cost)

            ELSE 0
        END
        AS DECIMAL(14,2)
    ) AS total_cost,

    -- Profit after discount and cost
    CAST(
        CASE
            WHEN order_status = 'Completed'
                THEN
                    (gross_revenue - discount_amount)
                    - (quantity * unit_cost)

            WHEN order_status = 'Returned'
                THEN
                    -(
                        (gross_revenue - discount_amount)
                        - (quantity * unit_cost)
                    )

            ELSE 0
        END
        AS DECIMAL(14,2)
    ) AS profit,

    -- Value of orders that were cancelled
    CAST(
        CASE
            WHEN order_status = 'Cancelled'
                THEN gross_revenue - discount_amount
            ELSE 0
        END
        AS DECIMAL(14,2)
    ) AS cancelled_value

FROM calculated;

num_affected_rows,num_inserted_rows


## Cell 3a - Add Business Metadata

**Description:**
Add column comments to improve discoverability in Genie and Databricks dashboards. These descriptions help users understand the business meaning of dimension keys, transaction attributes, and calculated measures.

In [0]:
%sql

-- ============================================================
-- CELL 3a: ADD BUSINESS METADATA TO SALES FACT TABLE
-- Purpose: Improve discoverability for Genie and dashboards
-- ============================================================

-- Dimension Foreign Keys
ALTER TABLE `end-to-end_pipeline`.gold.fact_sales 
  ALTER COLUMN transaction_id COMMENT 'Unique transaction identifier (business key for sales fact)';

ALTER TABLE `end-to-end_pipeline`.gold.fact_sales 
  ALTER COLUMN order_id COMMENT 'Order identifier grouping multiple transaction lines';

ALTER TABLE `end-to-end_pipeline`.gold.fact_sales 
  ALTER COLUMN customer_id COMMENT 'Foreign key to dim_customer (who purchased)';

ALTER TABLE `end-to-end_pipeline`.gold.fact_sales 
  ALTER COLUMN product_id COMMENT 'Foreign key to dim_product (what was purchased)';

ALTER TABLE `end-to-end_pipeline`.gold.fact_sales 
  ALTER COLUMN store_id COMMENT 'Foreign key to dim_store (where the sale occurred)';

ALTER TABLE `end-to-end_pipeline`.gold.fact_sales 
  ALTER COLUMN order_date COMMENT 'Foreign key to dim_date (when the order was placed)';

-- Transaction Attributes
ALTER TABLE `end-to-end_pipeline`.gold.fact_sales 
  ALTER COLUMN quantity COMMENT 'Number of units purchased in this transaction';

ALTER TABLE `end-to-end_pipeline`.gold.fact_sales 
  ALTER COLUMN unit_price COMMENT 'Price per unit before discount';

ALTER TABLE `end-to-end_pipeline`.gold.fact_sales 
  ALTER COLUMN unit_cost COMMENT 'Cost per unit for COGS calculation';

ALTER TABLE `end-to-end_pipeline`.gold.fact_sales 
  ALTER COLUMN discount_pct COMMENT 'Discount percentage applied to this transaction (0.0 to 1.0)';

ALTER TABLE `end-to-end_pipeline`.gold.fact_sales 
  ALTER COLUMN order_status COMMENT 'Transaction status (Completed, Returned, Cancelled)';

ALTER TABLE `end-to-end_pipeline`.gold.fact_sales 
  ALTER COLUMN payment_method COMMENT 'Payment method used for this transaction';

-- Calculated Business Measures
ALTER TABLE `end-to-end_pipeline`.gold.fact_sales 
  ALTER COLUMN gross_revenue COMMENT 'Total revenue before discount (quantity × unit_price)';

ALTER TABLE `end-to-end_pipeline`.gold.fact_sales 
  ALTER COLUMN discount_amount COMMENT 'Monetary value of discount applied (gross_revenue × discount_pct)';

ALTER TABLE `end-to-end_pipeline`.gold.fact_sales 
  ALTER COLUMN net_revenue COMMENT 'Revenue after discount; Completed = positive, Returned = negative, Cancelled = 0';

ALTER TABLE `end-to-end_pipeline`.gold.fact_sales 
  ALTER COLUMN total_cost COMMENT 'Cost of goods sold; Completed = positive, Returned = negative, Cancelled = 0';

ALTER TABLE `end-to-end_pipeline`.gold.fact_sales 
  ALTER COLUMN profit COMMENT 'Profit after discount and cost (net_revenue - total_cost)';

ALTER TABLE `end-to-end_pipeline`.gold.fact_sales 
  ALTER COLUMN cancelled_value COMMENT 'Value of cancelled transactions (tracked separately from net_revenue)';

#Cell 4 - Validate Gold Sales Fact

**Description:**
Confirm that the Gold Fact table preserves the transaction grain, contains valid business measures, matches the Silver source row count, and reports any dimension-key relationship issues.

In [0]:
%sql

-- ============================================================
-- CELL 4: VALIDATE GOLD SALES FACT
-- Purpose: Confirm transaction grain, measure integrity,
--          Silver → Gold completeness,
--          and dimension relationship quality
-- ============================================================

WITH fact_validation AS (

    SELECT
        COUNT(*) AS total_rows,

        COUNT(DISTINCT transaction_id)
            AS distinct_transaction_ids,

        COUNT(*) - COUNT(DISTINCT transaction_id)
            AS duplicate_transaction_ids,

        SUM(CASE WHEN transaction_id IS NULL THEN 1 ELSE 0 END)
            AS null_transaction_ids,

        SUM(CASE WHEN order_date IS NULL THEN 1 ELSE 0 END)
            AS null_order_dates,

        SUM(CASE WHEN gross_revenue < 0 THEN 1 ELSE 0 END)
            AS invalid_gross_revenue,

        SUM(CASE WHEN discount_amount < 0 THEN 1 ELSE 0 END)
            AS invalid_discount_amount,

        SUM(
            CASE
                WHEN order_status = 'Cancelled'
                     AND net_revenue != 0
                THEN 1
                ELSE 0
            END
        ) AS invalid_cancelled_revenue

    FROM `end-to-end_pipeline`.gold.fact_sales
),

source_check AS (

    SELECT
        COUNT(*) AS silver_rows

    FROM `end-to-end_pipeline`.silver.sales_transactions
),

relationship_check AS (

    SELECT

        SUM(CASE WHEN c.customer_id IS NULL THEN 1 ELSE 0 END)
            AS unmatched_customer_keys,

        SUM(CASE WHEN p.product_id IS NULL THEN 1 ELSE 0 END)
            AS unmatched_product_keys,

        SUM(CASE WHEN st.store_id IS NULL THEN 1 ELSE 0 END)
            AS unmatched_store_keys,

        SUM(CASE WHEN d.date IS NULL THEN 1 ELSE 0 END)
            AS unmatched_date_keys

    FROM `end-to-end_pipeline`.gold.fact_sales f

    LEFT JOIN `end-to-end_pipeline`.gold.dim_customer c
        ON f.customer_id = c.customer_id

    LEFT JOIN `end-to-end_pipeline`.gold.dim_product p
        ON f.product_id = p.product_id

    LEFT JOIN `end-to-end_pipeline`.gold.dim_store st
        ON f.store_id = st.store_id

    LEFT JOIN `end-to-end_pipeline`.gold.dim_date d
        ON f.order_date = d.date
)

SELECT
    f.*,
    s.silver_rows,

    r.unmatched_customer_keys,
    r.unmatched_product_keys,
    r.unmatched_store_keys,
    r.unmatched_date_keys,

    CASE
        WHEN f.total_rows = s.silver_rows
            AND f.total_rows = f.distinct_transaction_ids
            AND f.duplicate_transaction_ids = 0
            AND f.null_transaction_ids = 0
            AND f.null_order_dates = 0
            AND f.invalid_gross_revenue = 0
            AND f.invalid_discount_amount = 0
            AND f.invalid_cancelled_revenue = 0
        THEN 'PASS'
        ELSE 'FAIL'
    END AS validation_status

FROM fact_validation f
CROSS JOIN source_check s
CROSS JOIN relationship_check r;

total_rows,distinct_transaction_ids,duplicate_transaction_ids,null_transaction_ids,null_order_dates,invalid_gross_revenue,invalid_discount_amount,invalid_cancelled_revenue,silver_rows,unmatched_customer_keys,unmatched_product_keys,unmatched_store_keys,unmatched_date_keys,validation_status
24905,24905,0,0,0,0,0,0,24905,3,104,0,0,PASS
